# 05 -- Topic Modelling

Applies two complementary topic models to the tweet corpus:

| Model | Approach | Best for |
|---|---|---|
| **LDA** (Gensim) | Bag-of-words + Dirichlet priors | Interpretable topics, coherence scoring |
| **BERTopic** | Sentence embeddings + HDBSCAN clustering | Contextual topics, outlier handling |

The notebook ends by merging BERTopic labels with sentiment scores to produce
`topic_drift.csv`, the primary input for `07_drift_analysis.ipynb`.

**Input:** `DATA_DIR/tweets.csv`, `DATA_DIR/sentiment_results.csv`
**Output:** `DATA_DIR/topicmodel_tweets.csv`, `DATA_DIR/topic_drift.csv`


## Dependencies

In [ ]:
# Run once -- safe to skip if already installed
# !pip install bertopic umap-learn hdbscan gensim pyLDAvis


## Imports

In [ ]:
import re
import string
import warnings
from collections import Counter

import gensim
import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis
import seaborn as sns
from bertopic import BERTopic
from gensim import corpora
from gensim.models.coherencemodel import CoherenceModel
from gensim.models.ldamodel import LdaModel
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

warnings.filterwarnings('ignore')

# Download NLTK data (safe to re-run)
for pkg in ['wordnet', 'punkt', 'stopwords', 'punkt_tab', 'omw-1.4']:
    nltk.download(pkg, quiet=True)


## Config

Set `DATA_DIR` to the folder produced by `01_eda.ipynb`.
LDA and BERTopic hyperparameters can be tuned here.


In [ ]:
DATA_DIR = "data"   # <- change to your local path

INPUT_TWEETS    = f"{DATA_DIR}/tweets.csv"
INPUT_SENTIMENT = f"{DATA_DIR}/sentiment_results.csv"
OUTPUT_TOPICS   = f"{DATA_DIR}/topicmodel_tweets.csv"
OUTPUT_DRIFT    = f"{DATA_DIR}/topic_drift.csv"

# LDA hyperparameters
LDA_NUM_TOPICS  = 8
LDA_PASSES      = 10
LDA_NO_BELOW    = 10    # ignore tokens in fewer than this many docs
LDA_NO_ABOVE    = 0.5   # ignore tokens in more than this fraction of docs
LDA_RANDOM_SEED = 42

# Preprocessing
MIN_TOKENS      = 7     # drop tweets shorter than this after cleaning


## Load data

In [ ]:
df = pd.read_csv(INPUT_TWEETS)
df["input"] = df["tweet"].astype(str)
print(f"Loaded {len(df):,} tweets")
df.head()


## Preprocessing

`preprocess_text` lowercases, removes punctuation, strips stopwords,
and lemmatises. Tweets that produce fewer than `MIN_TOKENS` tokens are
dropped -- they carry too little signal for topic modelling.


In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


def preprocess_text(text: str) -> str:
    """
    Clean and normalise a tweet for topic modelling.

    Steps: lowercase -> strip punctuation -> tokenise ->
           remove stopwords -> lemmatise -> length filter.
    Returns an empty string for tweets that are too short after cleaning.
    """
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    if len(tokens) < MIN_TOKENS:
        return ""
    return " ".join(tokens)


df["input"] = df["input"].apply(preprocess_text)
df["tokens"] = df["input"].apply(word_tokenize)
df = df[df["input"].str.strip() != ""].reset_index(drop=True)

print(f"After preprocessing: {len(df):,} tweets retained")


## LDA topic model

Uses Gensim's `LdaModel` with a bag-of-words corpus. The dictionary is
filtered to remove very rare (`no_below`) and very common (`no_above`) terms
before training -- this improves topic quality significantly.


In [ ]:
# Build dictionary and BoW corpus
dictionary = corpora.Dictionary(df['tokens'])
dictionary.filter_extremes(no_below=LDA_NO_BELOW, no_above=LDA_NO_ABOVE)
corpus = [dictionary.doc2bow(tokens) for tokens in df['tokens']]

print(f'Vocabulary size after filtering: {len(dictionary):,} tokens')
print(f'Corpus size: {len(corpus):,} documents')


In [ ]:
lda_model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=LDA_NUM_TOPICS,
    random_state=LDA_RANDOM_SEED,
    passes=LDA_PASSES,
)

print('Top words per topic:')
for idx, topic in lda_model.print_topics(-1):
    print(f'  Topic {idx + 1}: {topic}')


In [ ]:
# Assign dominant topic to each document
doc_topics = [
    max(lda_model.get_document_topics(doc), key=lambda x: x[1])[0]
    for doc in corpus
]
df['lda_topic'] = doc_topics

plt.figure(figsize=(10, 5))
sns.countplot(x=doc_topics, color='steelblue')
plt.title('LDA -- Topic Distribution Across Documents')
plt.xlabel('Topic ID')
plt.ylabel('Number of Documents')
plt.tight_layout()
plt.show()


## LDA evaluation

**Coherence (C_v):** measures how often the top words of a topic co-occur in the
corpus. Higher is better; values above 0.5 are generally considered reasonable.

**Perplexity:** log-likelihood per token on held-out data. Lower is better,
but coherence is the more informative metric for human interpretability.


In [ ]:
coherence_model = CoherenceModel(
    model=lda_model,
    texts=df['tokens'].tolist(),
    dictionary=dictionary,
    coherence='c_v',
)
coherence_score = coherence_model.get_coherence()
perplexity_score = lda_model.log_perplexity(corpus)

print(f'LDA Coherence Score (C_v) : {coherence_score:.4f}')
print(f'LDA Log-Perplexity        : {perplexity_score:.4f}')
print()
print('Note: C_v > 0.5 is generally good. Perplexity is lower = better,\n'
      'but coherence is the more interpretable metric for topic quality.')


In [ ]:
# Interactive pyLDAvis visualisation
# Shows the topic positions (inter-topic distance map) and top-30 terms per topic
vis = gensimvis.prepare(lda_model, corpus, dictionary)
pyLDAvis.display(vis)


## BERTopic

BERTopic uses sentence embeddings (all-MiniLM-L6-v2) to cluster tweets into
topics via HDBSCAN, then extracts topic keywords with c-TF-IDF.

Topic `-1` is the outlier cluster -- tweets that didn't fit any coherent topic.
A large outlier cluster may indicate the corpus is too diverse or `MIN_TOKENS`
should be lowered.


In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

topic_model = BERTopic(
    language='english',
    calculate_probabilities=True,
    verbose=True,
    embedding_model=embedding_model,
    vectorizer_model=CountVectorizer(stop_words='english', min_df=1),
)

docs = df['input'].tolist()
topics, probs = topic_model.fit_transform(docs)
df['bertopic_topic'] = topics

print(f'\nNumber of topics found (excl. outliers): {topic_model.get_topic_info()["Topic"].nunique() - 1}')
print(f'Outlier tweets (topic -1): {(pd.Series(topics) == -1).sum():,}')


In [ ]:
# Top 10 topics by document count
display(topic_model.get_topic_info().head(10))

# Bar chart of top words per topic
topic_model.visualize_barchart(top_n_topics=10)


In [ ]:
# 2D inter-topic distance map
topic_model.visualize_topics()


## Attach topic names

BERTopic auto-generates a name for each topic from its top keywords.
These are added as a human-readable `bertopic_topic_name` column.


In [ ]:
topic_names = topic_model.get_topic_info().set_index('Topic')['Name'].to_dict()
df['bertopic_topic_name'] = df['bertopic_topic'].map(topic_names)

print('Sample topic name assignments:')
display(df[['tweet', 'bertopic_topic', 'bertopic_topic_name']].head(10))


## Token length distribution

A quick sanity check: the distribution of token counts after preprocessing.
A heavily right-skewed distribution is typical for social media text.


In [ ]:
df['length'] = df['tokens'].apply(len)

plt.figure(figsize=(10, 5))
sns.histplot(df['length'], bins=30, kde=True, color='steelblue')
plt.title('Distribution of Token Counts (after preprocessing)')
plt.xlabel('Token Count')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

print(df['length'].describe().round(2))


## Save topic model results

In [ ]:
df.to_csv(OUTPUT_TOPICS, index=False)
print(f'Saved {len(df):,} rows -> {OUTPUT_TOPICS}')


## Build topic_drift.csv

Merges BERTopic labels with the RoBERTa sentiment labels from
`02_sentiment_emotion.ipynb` to produce a single CSV used by
`07_drift_analysis.ipynb`. Only the columns needed for drift analysis
are retained to keep the file lean.


In [ ]:
sentiment = pd.read_csv(INPUT_SENTIMENT)[['id', 'roberta_label']]
sentiment = sentiment.rename(columns={'roberta_label': 'sentiment'})

topic_drift = df.merge(sentiment, on='id', how='left')
topic_drift = topic_drift[[
    'id', 'tweet', 'user_id', 'comment_to', 'thread_id',
    'round', 'day', 'hour',
    'bertopic_topic', 'bertopic_topic_name',
    'length', 'sentiment',
]]

topic_drift.to_csv(OUTPUT_DRIFT, index=False)
print(f'Saved {len(topic_drift):,} rows -> {OUTPUT_DRIFT}')
topic_drift.head()
